In [23]:
import requests
from lxml import etree
import pandas as pd
import re
from bs4 import BeautifulSoup
import numpy as np

In [2]:
url='https://www.cryst.ehu.es/magndata/search.php?show_db=1'
headers={
"User-Agent": 'Mozilla/4.0 (compatible; MSIE 6.0; Windows NT 5.1; SV1; AcooBrowser; .NET CLR 1.1.4322; .NET CLR 2.0.50727)'
                } 
response = requests.get(url,headers=headers)
c=response.status_code
print(c)
html_data=response.text.replace('<sub>', '').replace('</sub>', '')
# print(data)

200


In [3]:
magnetic_elements=['la','ce','pr','nd','pm','sm','eu','gd','tb','dy','ho','er','tm','yb','lu']

In [4]:
tree = etree.HTML(html_data)
element_names=tree.xpath('//tr/td[@width="200" and @align="center" and @valign="top"]')

In [27]:
nu1=tree.xpath(r"//a[contains(@href, 'index.php?index=') and @class='blue']")
all_elements=[[nu.xpath("./text()[1]")[0],nu.xpath(r"./@href")[0]] for nu in nu1]
print(len(all_elements))
# name_url=nu.xpath(r"./@href")
# name = nu.xpath("./text()[1]")
print(all_elements)

2348
[['LaMnO3', 'index.php?index=0.1'], ['Cd2Os2O7', 'index.php?index=0.2'], ['Ca3LiOsO6', 'index.php?index=0.3'], ['NiCr2O4', 'index.php?index=0.4'], ['Cr2S3', 'index.php?index=0.5'], ['YMnO3', 'index.php?index=0.6'], ['ScMnO3', 'index.php?index=0.7'], ['ScMnO3', 'index.php?index=0.8'], ['GdB4', 'index.php?index=0.9'], ['DyFeO3', 'index.php?index=0.10'], ['DyFeO3', 'index.php?index=0.11'], ['U3Ru4Al12', 'index.php?index=0.12'], ['Ca3Co2-xMnxO6', 'index.php?index=0.13'], ['Gd5Ge4', 'index.php?index=0.14'], ['MnF2', 'index.php?index=0.15'], ['EuTiO3', 'index.php?index=0.16'], ['FePO4', 'index.php?index=0.17'], ['BaMn2As2', 'index.php?index=0.18'], ['MnTiO3', 'index.php?index=0.19'], ['MnTe2', 'index.php?index=0.20'], ['PbNiO3', 'index.php?index=0.21'], ['DyB4', 'index.php?index=0.22'], ['Ca3Mn2O7', 'index.php?index=0.23'], ['LiMnPO4', 'index.php?index=0.24'], ['NaOsO3', 'index.php?index=0.25'], ['TmAgGe', 'index.php?index=0.26'], ['YFe4Ge2', 'index.php?index=0.27'], ['LiFeSi2O6', 'inde

In [ ]:
# all_elements=[]
# print(len(element_names))
# for td in element_names:
#     td_html = etree.tostring(td, encoding='utf-8', method='html').decode('utf-8')
#     cleaned_str = td_html.replace('<sub>', '').replace('</sub>', '')
#     td_tree = etree.HTML(cleaned_str)
#     name = td_tree.xpath(".//a[contains(@href, 'index.php?index=') and @class='blue']/text()")
#     name_url=td_tree.xpath(".//a[contains(@href, 'index.php?index=') and @class='blue']/@href")
#     all_elements.append(name+name_url)
# print(all_elements)
# print(len(all_elements))
#     

In [35]:
added_indexes = set()  # 记录已添加的index
useful_elements = []

for item in all_elements:
    compound = item[0].lower()
    if any(n in compound for n in magnetic_elements):
        # 用index作为唯一标识去重
        if item[1] not in added_indexes:
            useful_elements.append(item)
            added_indexes.add(item[1])

In [36]:
print(len(useful_elements))
print(useful_elements)

1071
[['LaMnO3', 'index.php?index=0.1'], ['GdB4', 'index.php?index=0.9'], ['DyFeO3', 'index.php?index=0.10'], ['DyFeO3', 'index.php?index=0.11'], ['Gd5Ge4', 'index.php?index=0.14'], ['EuTiO3', 'index.php?index=0.16'], ['DyB4', 'index.php?index=0.22'], ['TmAgGe', 'index.php?index=0.26'], ['Er2Ti2O7', 'index.php?index=0.29'], ['YbMnO3', 'index.php?index=0.30'], ['HoMnO3', 'index.php?index=0.31'], ['HoMnO3', 'index.php?index=0.32'], ['HoMnO3', 'index.php?index=0.33'], ['La0.5Sr0.5FeO2.5F0.5', 'index.php?index=0.34'], ['Nd2NaRuO6', 'index.php?index=0.39'], ['HoMnO3', 'index.php?index=0.42'], ['HoMnO3', 'index.php?index=0.43'], ['La2NiO4', 'index.php?index=0.45'], ['Gd2Sn2O7', 'index.php?index=0.47'], ['Tb2Sn2O7', 'index.php?index=0.48'], ['Ho2Ru2O7', 'index.php?index=0.49'], ['Ho2Ru2O7', 'index.php?index=0.51'], ['Ho2CrSbO7', 'index.php?index=0.63'], ['Tb2Ti2O7', 'index.php?index=0.77'], ['Gd2CuO4', 'index.php?index=0.82'], ['YBaMn2O5.5', 'index.php?index=0.98'], ['YBaMn2O5.5', 'index.php?

In [37]:
index_q=[i[1] for i in useful_elements]
arr=np.array(index_q)
unique_elements,counts = np.unique(arr, return_counts=True)
duplicates = unique_elements[counts>=2]
print(duplicates)

[]


In [38]:
def delete_sub(html_content,tags_to_remove:list):
    """
    删除HTML标签中的<sub>标签
    """
    for tag in tags_to_remove:
    # 构建正则表达式，匹配开标签和闭标签
        pattern = re.compile(f'<\/?{tag}>', re.IGNORECASE)
        html_content = pattern.sub('', html_content)
        
    return html_content

In [39]:
def search_name(single_tree:etree):
# 假设single_tree是lxml解析的对象，先获取所有符合条件的form标签
    form_tags = single_tree.xpath("//td/form[@target='_blank']")
# 取第一个form标签并转换为字符串
    form_str = etree.tostring(form_tags[0], encoding='utf-8', method='html').decode('utf-8')

# 使用BeautifulSoup解析
    soup = BeautifulSoup(form_str, 'html.parser')

# 删除所有sub和small标签
    for tag in soup.find_all(['sub', 'small']):
        tag.unwrap()  # 完全移除标签及其内容
    new_name = [h2.get_text() for h2 in soup.find_all('h2')]
    return new_name[0]

In [40]:
def search_table(target_table_string:str):
    table=[]
    # cleaned_target_table_string = delete_sub(target_table_string, ['sub','small'])
    cleaned_target_table=BeautifulSoup(target_table_string, 'html.parser')
    for tag in cleaned_target_table.find_all(['sub', 'font','b']):
        tag.unwrap()
    tr_table=cleaned_target_table.find_all('tr')
    headers1 = [th.text for th in tr_table[0].find_all('th')]
    table.append(headers1)
    data=[]
    for row in tr_table[1:]:
        row_data=[td.text for td in row.find_all('td')]
        data.append(row_data)
    table.append(data)
    return  table

In [41]:
def seach_vectors(single_tree:etree):
    Propagation_vector=single_tree.xpath("//b[contains(text(), 'Propagation vector:')]/following-sibling::text()[1]")
    if Propagation_vector:
        print(Propagation_vector)
        return Propagation_vector
    else:
        Propagation_vector=single_tree.xpath("//br[preceding::b[contains(text(), 'Propagation vector(s):')] and following::b[contains(text(), 'Transition Temperature:')]]") 
        target_brs=Propagation_vector[:-2]
        Propagation_vector = [br.xpath("./following-sibling::text()[1]") for br in target_brs]
        print(Propagation_vector)
        return Propagation_vector

In [42]:
def web_3(single_tree, response_1):

    form_action = single_tree.xpath("//form[@method='post']/@action")
    action_url = fr"https://www.cryst.ehu.es{form_action[0]}"
    mcif_value = single_tree.xpath("//form/input[@name='mcif']/@value")
    mcif = mcif_value[1]
    print(mcif)
    form_data = {
        "mcif": mcif,  # 提取的mcif值
        "choose": "Get_mirreps",  # 提交按钮值
        "mode": "irreps"  # 新增的mode参数
    }
    post_response = requests.post(action_url, data=form_data, cookies=response_1.cookies,timeout=60)
    post_response.encoding = "utf-8"
    ptext = post_response.text
    print(ptext)
    single_tree = etree.HTML(ptext)
    data_value = single_tree.xpath('//meta[@http-equiv="REFRESH"]/@content')[0][6:]
    return data_value

In [13]:
useful_elements=[['Eu3PbO', 'index_incomm.php?index=1.1.16']]

In [43]:
useful_elements = [['Eu3PbO', 'index.php?index=0.271']]

In [44]:
for i in useful_elements:
    single_url=r'https://www.cryst.ehu.es/magndata/'+f'{i[1]}'
    print(single_url)
    response_1 = requests.get(single_url,headers=headers)
    c_1=response_1.status_code
    html_data_1=response_1.text
    # print(html_data_1)
    single_tree=etree.HTML(html_data_1)
    # Propagation_vector= seach_vectors(single_tree)
    url_g=web_3(single_tree, response_1)
    print(url_g)
    # print(html_data_1)

https://www.cryst.ehu.es/magndata/index.php?index=0.271
I1wjQ0lGXzIuMAojIENyZWF0ZWQgYnkgdGhlIEJpbGJhbyBDcnlzdGFsbG9ncmFwaGljIFNlcnZlcgojIGh0dHA6Ly93d3cuY3J5c3QuZWh1LmVzCiMgRGF0ZTogMjkvMTIvMjAxOQoKZGF0YV81eU9odEFvUgpfYXVkaXRfY3JlYXRpb25fZGF0ZSAgICAgICAgICAgIDIwMTktMTItMjkKX2F1ZGl0X2NyZWF0aW9uX21ldGhvZCAgICAgICAgICAiQmlsYmFvIENyeXN0YWxsb2dyYXBoaWMgU2VydmVyIgoKX2NpdGF0aW9uX2pvdXJuYWxfYWJicmV2ICAgICAgICAiUGh5cy4gUmV2LiBCIgpfY2l0YXRpb25fam91cm5hbF92b2x1bWUgICAgICAgIDk5Cl9jaXRhdGlvbl9wYWdlX2ZpcnN0ICAgICAgICAgICAgPwpfY2l0YXRpb25fcGFnZV9sYXN0ICAgICAgICAgICAgID8KX2NpdGF0aW9uX2FydGljbGVfaWQgICAgICAgICAgICAxODQ0NDQKX2NpdGF0aW9uX3llYXIgICAgICAgICAgICAgICAgICAyMDE5Cl9jaXRhdGlvbl9ET0kgICAgICAgICAgICAgICAgICAgMTAuMTEwMy9QaHlzUmV2Qi45OS4xODQ0NDQKCmxvb3BfCl9jaXRhdGlvbl9hdXRob3JfbmFtZQoiSi5MLiBHYXJjaWEtTXVub3oiCiJKLiBCbGFzY28iCiJYLiBaaGFuZyIKIk8uIEZhYmVsbyIKCgoKCl9hdG9taWNfcG9zaXRpb25zX3NvdXJjZV9kYXRhYmFzZV9jb2RlX0lDU0QgIC4KX2F0b21pY19wb3NpdGlvbnNfc291cmNlX290aGVyICAgICAgICAgICAgICAgLgoKX05lZWxfdGVtcGVy

In [46]:
def index_incomm_set(html_data_1:str):
    single_tree=etree.HTML(html_data_1)
    new_url=single_tree.xpath("//head/meta[@http-equiv='REFRESH']/@content")
    if new_url:
        url = new_url[0].split('url=')[1]
        return url
    else:
        return False


In [47]:
for i in useful_elements:
    single_url=r'https://www.cryst.ehu.es/magndata/'+f'{i[1]}'
    print(single_url)
    response_1 = requests.get(single_url,headers=headers)
    c_1=response_1.status_code
    html_data_1=response_1.text
    # print(html_data_1)
    single_tree=etree.HTML(html_data_1)
    new_name=search_name(single_tree)

    print(new_name)
    img_url_1=single_tree.xpath("//i[contains(text(), 'Magnetic structure with all atoms')]/../..//a/@href")
    #/../..：从 <i> 向上回溯两次，到达共同的父节点 <td>（第一次 .. 到 <br>，第二次 .. 到 <td>）。
    parent_space_group_url=single_tree.xpath("//b[contains(text(), 'Parent space group')]/following::a[@class='blue'][1]/@href")
    Propagation_vector=single_tree.xpath("//b[contains(text(), 'Propagation vector:')]/following-sibling::text()[1]")
    Transition_Temperature=single_tree.xpath("//b[contains(text(), 'Transition Temperature:')]/following-sibling::text()[1]")
    Experiment_Temperature=single_tree.xpath("//b[contains(text(), 'Experiment Temperature:')]/following-sibling::text()[1]")
    Lattice_parameters_of_the_magnetic_unit_cell=single_tree.xpath("//b[contains(text(), 'Lattice parameters of the magnetic unit cell:')]/following-sibling::text()[1]")
    BNS_Magnetic_Space_Group=single_tree.xpath("//b[contains(text(), 'BNS Magnetic Space Group:')]/following::a[@class='blue'][1]/@href")
    print(img_url_1)
    print(parent_space_group_url)
    print(Propagation_vector)
    print(Transition_Temperature)
    print(Experiment_Temperature)
    print(Lattice_parameters_of_the_magnetic_unit_cell)
    print(BNS_Magnetic_Space_Group)
    mcif_value = single_tree.xpath("//form/input[@name='mcif']/@value")
    mcif = mcif_value[0]   
    form_action = single_tree.xpath("//form[@method='post']/@action")
    action_url = fr"https://www.cryst.ehu.es{form_action[0]}"
    # print(action_url)
    
    #Magnetic atoms
    target_table1 = single_tree.xpath("//font[contains(text(), 'Magnetic atoms')]/following::table[@class='sample'][1]")
    if not target_table1:
        print("未找到目标表格，请检查HTML结构或XPath")
    else:
        target_table_string=etree.tostring(target_table1[0], encoding='utf-8', method='html').decode('utf-8')
        
        # cleaned_target_table_string = delete_sub(target_table_string, ['sub','small'])
        cleaned_target_table=BeautifulSoup(target_table_string, 'html.parser')
        for tag in cleaned_target_table.find_all(['sub', 'font','b']):
            tag.unwrap()
        tr_table=cleaned_target_table.find_all('tr')
       
        headers1 = [th.text for th in tr_table[0].find_all('th')]
        # print(headers1)
        data=[]
        for row in tr_table[1:]:
            row_data=[td.text for td in row.find_all('td')]
            data.append(row_data)
        # print(data)
     
        # 5. 构造DataFrame
        if headers1 and data:
            df = pd.DataFrame(data, columns=headers1)
            print("提取的DataFrame<Magnetic atoms>：")
            print(df)
        else:
            print("表头或数据为空，提取失败")
            
        p1_tags = single_tree.xpath("//p/b/font[@color='red']/text()")
        # print(f'ofsdjfklajsdlfj{len(p1_tags)}')
        # print(p1_tags)
        for i1 in p1_tags:
            if 'Set of atoms in the unit cell related by symmetry with the magnetic atom' in i1:
                target_table1 = single_tree.xpath(f"//font[contains(text(), '{i1}')]/following::table[@class='sample'][1]")
                target_table_string=etree.tostring(target_table1[0], encoding='utf-8', method='html').decode('utf-8')
                table_1=search_table(target_table_string)
                df = pd.DataFrame(table_1[1], columns=table_1[0])
                print(f"提取的DataFrame<{i1}>：")
                print(df)
    # Non-magnetic atoms
    target_table = single_tree.xpath("//b[contains(text(), 'Non-magnetic atoms')]/following::table[@class='sample'][1]")
    if not target_table:
        print("未找到目标表格，请检查HTML结构或XPath")
    else:
        target_table = target_table[0]  # 获取表格节点
        # 2. 提取表头（表格第一行的<th>）
        headers1 = target_table.xpath(".//tr[1]/th/text()")
        # 3. 提取数据行（表格tbody中除第一行外的<tr>）
        rows = target_table.xpath(".//tr[position() > 1]")
        # 4. 遍历行，提取单元格文本
        data = []
        for row in rows:
            cell_texts = row.xpath("./td/text()")
            data.append(cell_texts)

        # 5. 构造DataFrame
        if headers1 and data:
            df = pd.DataFrame(data, columns=headers1)
            print("提取的DataFrame<Non-magnetic atoms>：")
            print(df)
        else:
            print("表头或数据为空，提取失败")  
    p_tags=single_tree.xpath("//p/text()")
    # print(p_tags)
    for i_1 in p_tags:
        if 'Set of atoms in the unit cell related by symmetry with the atom' in i_1:
            target_table = single_tree.xpath(f"//p[contains(text(), '{i_1}')]/following::table[@class='sample'][1]")
            target_table = target_table[0]  # 获取表格节点
            # 2. 提取表头（表格第一行的<th>）
            headers1 = target_table.xpath(".//tr[1]/th/text()")
            # 3. 提取数据行（表格tbody中除第一行外的<tr>）
            rows = target_table.xpath(".//tr[position() > 1]")
            # 4. 遍历行，提取单元格文本
            data = []
            for row in rows:
                cell_texts = row.xpath("./td/text()")
                data.append(cell_texts)
     
            # 5. 构造DataFrame
            if headers1 and data:
                df = pd.DataFrame(data, columns=headers1)
                print(f"提取的DataFrame《{i_1}》：")
                print(df)
            else:
                print("表头或数据为空，提取失败")   
        

https://www.cryst.ehu.es/magndata/index.php?index=0.271
Tb2MnNiO6 (#0.271)
['dbfiles/Tb2MnNiO6_0.271/0.271.Tb2MnNiO6.png', '']
['/cgi-bin/cryst/programs/nph-getgen?list=new&what=gen&gnum=14']
['k1 (0, 0, 0)']
['120 K']
['75 K']
['\n5.2728 5.5181 7.5104 90.00 90.165 90.00']
['https://www.cryst.ehu.es/cgi-bin/cryst/programs/nph-magtrgen?gnum=14.79']
提取的DataFrame<Magnetic atoms>：
  Label Atom type        x        y        z Multiplicity  \
0   Tb1        Tb  0.01700  0.06600  0.74900            4   
1   Ni1        Ni  0.50000  0.00000  0.50000            2   
2   Mn1        Mn  0.50000  0.00000  0.00000            2   

  Symmetry constraints on M       Mx   My       Mz   |M|  
0                  mx,my,mz      0.0  0.0      0.0  0.00  
1                  mx,my,mz  0.56(6)  0.0  1.23(3)  1.35  
2                  mx,my,mz  0.85(8)  0.0  1.85(4)  2.03  
提取的DataFrame<Set of atoms in the unit cell related by symmetry with the magnetic atom Tb1:>：
  Atom        x        y        z Symmetry con

In [ ]:
base_url = "http://bilbao-cryst.ehu.es"  # 晶体学常用数据库域名
form_page_url = f"{base_url}/cgi-bin/cryst/programs/nph-nagndata"  
print(form_page_url)

In [68]:
print(str([]))

[]
